# Evaluation Pipeline

In [64]:
import os
import torch
import cv2
from scipy.stats import entropy
import numpy as np
import warnings
import csv
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:0


input which dataset should be evaluated

In [ ]:
image_type = "all" # must be in: all, desktop, mobile, poster, web
seconds = "1s" # must be in: 1s, 3s, 7s
architectur = "sparse" #must be in: rich, sparse


In [66]:
'''
A set of metrics for evaluatation of eye-gaze heatmaps,
adapted from UEyes Implenentation:
'''

EPS = np.finfo(np.float32).eps

def normalize(any_map):
    '''
    Normalize heatmap values in the [0,1] range.
    '''
    any_map = rescale(any_map)
    any_map /= (EPS + any_map.sum())
    return any_map


def whiten(any_map):
    '''
    Apply whitening algorithm.
    See https://en.wikipedia.org/wiki/Whitening_transformation
    '''
    return (any_map - any_map.mean()) / (EPS + any_map.std())

def rescale(any_map):
    '''
    Apply mimaxscaler algorithm.
    See https://en.wikipedia.org/wiki/Feature_scaling#Rescaling_(min-max_normalization)
    '''
    return (any_map - any_map.min()) / (EPS + any_map.max() - any_map.min())

def auc(sal_map, ref_map, fixation_threshold=0.7):
    '''
    Compute Judd's AUC score.
    '''
    def area_under_curve(predicted, actual, labelset):
        def roc_curve(predicted, actual, cls):
            si = np.argsort(-predicted)
            tp = np.cumsum(np.single(actual[si]==cls))
            fp = np.cumsum(np.single(actual[si]!=cls))
            tp = tp/np.sum(actual==cls)
            fp = fp/np.sum(actual!=cls)
            # print(predicted, actual, cls)
            tp = np.hstack((0.0, tp, 1.0))
            fp = np.hstack((0.0, fp, 1.0))

            return tp, fp

        # integration
        def auc_from_roc(tp, fp):
            # print(fp)
            h = np.diff(fp)
            # print(h,tp[1:]+tp[:-1])
            return np.sum(h*(tp[1:]+tp[:-1]))/2.0

        tp, fp = roc_curve(predicted, actual, np.max(labelset))
        return auc_from_roc(tp, fp)

    ref_map = (ref_map > fixation_threshold).astype(int)
    salShape = sal_map.shape
    fixShape = ref_map.shape
    # print(np.max(sal_map))
    
    predicted = sal_map.reshape(salShape[0]*salShape[1], -1, order='F').flatten()
    actual = ref_map.reshape(fixShape[0]*fixShape[1], -1, order='F').flatten()
    labelset = np.arange(2)

    return area_under_curve(predicted, actual, labelset)



def auc_shuff(sal_map, ref_map, rand_map=None, step_size=.01):
    '''
    Compute shuffled AUC score.
    '''
    sal_map -= np.min(sal_map)
    ref_map = np.vstack(np.where(ref_map != 0)).T

    if np.max(sal_map) > 0:
        sal_map = sal_map / np.max(sal_map)
    Sth = np.asarray([ sal_map[y-1][x-1] for y,x in ref_map ])
    Nfixations = len(ref_map)

    if rand_map is None:
        height, width = sal_map.shape
        rand_map = np.zeros((height, width))

    others = np.copy(rand_map)
    for y,x in ref_map:
        others[y-1][x-1] = 0

    ind = np.nonzero(others) # find fixation locations on other images
    nFix = rand_map[ind]
    randfix = sal_map[ind]
    Nothers = sum(nFix)

    allthreshes = np.arange(0,np.max(np.concatenate((Sth, randfix), axis=0)),step_size)
    allthreshes = allthreshes[::-1]
    tp = np.zeros(len(allthreshes)+2)
    fp = np.zeros(len(allthreshes)+2)
    tp[-1] = 1.0
    fp[-1] = 1.0
    tp[1:-1] = [float(np.sum(Sth >= thresh)) / Nfixations for thresh in allthreshes]
    fp[1:-1] = [float(np.sum(nFix[randfix >= thresh])) / Nothers for thresh in allthreshes]

    return np.trapz(tp,fp)


def nss(sal_map, ref_map):
    '''
    Compute Normalized Scanpath Saliency score.
    '''
    # print(sal_map)
    sal_map = whiten(sal_map)
    # print(sal_map)

    mask = ref_map.astype(np.bool_)
    # print(mask)
    return sal_map[mask].mean()


def infogain(sal_map, ref_map, rand_map=None):
    '''
    Compute InfoGain score.
    '''
    sal_map = normalize(sal_map)

    if rand_map is None:
        height, width = sal_map.shape
        rand_map = np.zeros((height, width))

    rand_map = normalize(rand_map)

    fixs = ref_map.astype(np.bool_)

    return (np.log2(EPS + sal_map[fixs]) \
          - np.log2(EPS + rand_map[fixs])).mean()


def similarity(sal_map, ref_map):
    '''
    Compute Similarity score.
    '''
    sal_map = normalize(sal_map)
    ref_map = normalize(ref_map)

    return np.minimum(sal_map, ref_map).sum()


def cc(sal_map, ref_map):
    '''
    Compute Coefficient of Correlation score.
    '''
    sal_map = whiten(sal_map)
    ref_map = whiten(ref_map)
    score = np.corrcoef(sal_map.flatten(), ref_map.flatten())
    return score[0][1]


def kldiv(sal_map, ref_map):
    '''
    Compute Kullback-Leibler divergence.
    '''
    return entropy(sal_map.flatten() + EPS, ref_map.flatten() + EPS)

def load_files(dirname, mode='dict', allow_ext=('.jpg','.jpeg','.png','.csv')):
    '''
    Read valid files in dir, according ot their file extension.
    Default allowed file extensions are: .png, .jpg, .jpeg, .csv
    '''
    img_list = []
    img_dict = {}
    for root, dirs, files in os.walk(dirname):
        for f in files:
            if not f.endswith(tuple(allow_ext)) or f in img_dict:
                continue

            img_path = os.path.join(root, f)
            if mode == 'dict':
                img_dict[f] = img_path
            else:
                img_list.append(img_path)

    if mode == 'dict':
        return img_dict
    return img_list

def mean_std(values):
    '''
    Compute mean and std of a list of values.
    '''
    v = np.array(values)
    return v.mean(), v.std()


In [67]:
pred_dir = f"results/{architectur}/{image_type}_{seconds}"
ref_dir = f"sorted/{image_type}"
ref_files = sorted(load_files(ref_dir, mode='list'))
pred_files = sorted(load_files(pred_dir, mode='list'))

# Initialize result arrays.
# Ensure we have exactly the same number of heatmaps to compare.
if len(ref_files) > len(pred_files):
    pred_dir = '/'.join(pred_files[0].split('/')[:-1]) + '/'
    ref_dir = '/'.join(ref_files[0].split('/')[:-1]) + '/'
    for f_path in ref_files:
        f_name = f_path.split('/')[-1]
        if pred_dir + f_name not in pred_files:
            print(ref_dir + f_name)
            ref_files.remove(ref_dir + f_name)

assert len(ref_files) == len(pred_files), 'The number of reference and prediction files do not match.'


a_auc, a_nss, a_inf, a_sim, a_cor, a_kld = [], [], [], [], [], []

for prediction, reference in zip(pred_files, ref_files):
    print(f'Comparing {prediction} vs {reference} heatmaps ...')

    sal_map = cv2.imread(prediction, 0)
    if np.max(sal_map) > 1:
        sal_map = np.divide(sal_map, np.max(sal_map))
    ref_map = cv2.imread(reference, 0)
    if np.max(ref_map) > 1:
        ref_map = np.divide(ref_map, np.max(ref_map))

    # Ensure that both heatmaps were produced over the same source image.
    if ref_map.shape != sal_map.shape:
        warnings.warn('The heatmap shapes do not match! Resizing ref_map to match sal_map.', stacklevel=2)
        ref_map = cv2.resize(ref_map.astype(np.float32), (sal_map.shape[1], sal_map.shape[0]), interpolation=cv2.INTER_LINEAR)

    auc_score = auc(sal_map, ref_map)
    nss_score = nss(sal_map, ref_map)
    inf_score = infogain(sal_map, ref_map)
    sim_score = similarity(sal_map, ref_map)
    cor_score = cc(sal_map, ref_map)
    kld_score = kldiv(sal_map, ref_map)

    # Finally store the values in the result arrays.
    a_auc.append(auc_score)
    a_nss.append(nss_score)
    a_inf.append(inf_score)
    a_sim.append(sim_score)
    a_cor.append(cor_score)
    a_kld.append(kld_score)

        # Finally report stats.
    stats = {
        'AUC (Judd)': mean_std(a_auc),
        'NSS':        mean_std(a_nss),
        'InfoGain':   mean_std(a_inf),
        'SIM':        mean_std(a_sim),
        'CC':         mean_std(a_cor),
        'KLDiv':      mean_std(a_kld),
    }

    print('AUC (judd) :', stats['AUC (Judd)'])
    print('       NSS :', stats['NSS'])
    print('  infogain :', stats['InfoGain'])
    print('similarity :', stats['SIM'])
    print('        CC :', stats['CC'])
    print('    KL div :', stats['KLDiv'])

Comparing results/rich/web_1s\014b22.png vs sorted/web\014b22.png heatmaps ...
AUC (judd) : (0.9901913897590451, 0.0)
       NSS : (0.6511903375877565, 0.0)
  infogain : (4.229545114038789, 0.0)
similarity : (0.44122314278719427, 0.0)
        CC : (0.49575239468066934, 0.0)
    KL div : (4.034021591228249, 0.0)
Comparing results/rich/web_1s\0af35a.png vs sorted/web\0af35a.png heatmaps ...
AUC (judd) : (0.9705610351461642, 0.01963035461288093)
       NSS : (0.8580511378323847, 0.20686080024462833)
  infogain : (4.13495251420169, 0.09459259983709867)
similarity : (0.3471194638370648, 0.09410367895012947)
        CC : (0.436745093593293, 0.05900730108737634)
    KL div : (6.184123161249403, 2.1501015700211537)
Comparing results/rich/web_1s\142b9c.png vs sorted/web\142b9c.png heatmaps ...
AUC (judd) : (0.9515035825877795, 0.03135719286195445)
       NSS : (0.8396190181353061, 0.17090078951920284)
  infogain : (4.032719387065926, 0.163915825544319)
similarity : (0.39038446727227943, 0.09822

In [68]:
os.makedirs('results', exist_ok=True)

csv_name = os.path.basename(os.path.normpath(f'resluts_{architectur}_{image_type}_{seconds}')) + '.csv'
csv_path = os.path.join('results', csv_name)

with open(csv_path, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['Metric', 'Mean', 'std'])  # header

    for metric, (mean, std) in stats.items():
        writer.writerow([metric, float(mean), float(std)])

print(f'Metrics saved to {csv_path}')

Metrics saved to results\resluts_rich_web_1s.csv
